# Hands-on RL Fine-tuning with verl on AMD Instinct™ MI300X

## Qwen3-4B LoRA GRPO Training on a Single GPU

In this workshop, we will run a small but complete RL fine-tuning workflow for a reasoning-style language model.

Modern LLMs are moving from chatbots to reasoning agents.  To support reasoning, tool use, and multi-step decision making, we use reinforcement learning after pretraining and supervised fine-tuning.

We will train **Qwen3-4B** with **LoRA + GRPO** using **verl** and **vLLM** on a single **AMD Instinct™ MI300X GPU**.

| Component | Role |
|---|---|
| **Qwen3-4B** | Base model |
| **LoRA** | Lightweight fine-tuning method |
| **GRPO** | Reward-based RL algorithm |
| **verl** | RL training framework |
| **vLLM** | Fast rollout generation engine |
| **AMD MI300X** | Single-GPU training platform |

## Why RL Workloads Fit AMD Instinct GPUs

Reinforcement learning workloads differ from ordinary supervised fine-tuning.

In RL post-training, the model repeatedly generates responses, scores those responses, and updates the policy. This means rollout generation can become a major part of the workload.

AMD Instinct MI300X is useful for this workshop because it provides:

- large HBM memory headroom
- high memory bandwidth
- strong compute capability
- ROCm software support
- enough room for model weights, LoRA adapters, rollout generation, and training overhead

This makes MI300X a strong fit for a single-GPU hands-on RL training demo.


## How verl and vLLM Work Together

In this training job, **verl controls the RL loop**, while **vLLM handles rollout generation**.

| Component | Responsibility |
|---|---|
| **verl** | Orchestrates training, reward calculation, GRPO update, LoRA update, logging, and checkpointing |
| **vLLM** | Generates multiple responses for each prompt efficiently |
| **GRPO** | Compares generated responses and reinforces better ones |
| **LoRA** | Updates small adapter weights instead of the full model |

### Runtime Flow

```text
Prompt
  ↓
vLLM generates multiple responses
  ↓
Reward function scores responses
  ↓
GRPO compares responses
  ↓
VERL updates LoRA adapters
  ↓
Next training step

## Workshop Runtime Stack

| Layer | Setting |
|---|---|
| Model | `Qwen/Qwen3-4B` |
| Dataset | GSM8K |
| RL algorithm | `algorithm.adv_estimator=grpo` |
| Fine-tuning | LoRA, `lora_rank=32`, `lora_alpha=64` |
| Rollout backend | `actor_rollout_ref.rollout.name=vllm` |
| Number of samples | `actor_rollout_ref.rollout.n=8` |
| Tensor parallel size | `tensor_model_parallel_size=1` |
| Framework version | verl v0.7.1 |
| Rollout runtime | vLLM 0.18.0 with ROCm 7.2.1 |
| Hardware | AMD Instinct MI300X, single GPU |


## Step 1: Check AMD GPU

Before running VERL, we first check whether the AMD GPU is visible.

Run `amd-smi` to confirm the GPU status.

In [ ]:
!amd-smi

## Step 2: Set Workshop Environment Variables

We use one GPU for this workshop.

The important settings are:

- `HIP_VISIBLE_DEVICES=0`: use the first visible AMD GPU
- `GPUS_PER_NODE=1`: keep the training setup single-GPU

In [ ]:
import os

os.environ["HIP_VISIBLE_DEVICES"] = "0"
os.environ["GPUS_PER_NODE"] = "1"

print("HIP_VISIBLE_DEVICES =", os.environ.get("HIP_VISIBLE_DEVICES"))
print("GPUS_PER_NODE =", os.environ.get("GPUS_PER_NODE"))

## Step 3: Install verl

This notebook assumes verl v0.7.1.

The workshop uses the verl training entrypoint:

```bash
python3 -m verl.trainer.main_ppo
```


In [ ]:
%%bash
set -e

WORKDIR="${HOME}/verl-workshop"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

if [ ! -d "verl" ]; then
  git clone https://github.com/verl-project/verl.git
fi

cd verl

# Use the workshop target version.
# If your environment uses a branch name such as release/v0.7.1, replace this checkout line accordingly.
git fetch --all --tags
git checkout v0.7.1 || git checkout release/v0.7.1

python3 -m pip install -U pip
python3 -m pip install -e .

python3 - <<'PY'
import verl
print("VERL import check: OK")
print("VERL module:", verl.__file__)
PY


## Step 4: Check vLLM and ROCm Runtime

In this workshop, verl uses vLLM as the rollout engine.

The key configuration later will be:

```bash
actor_rollout_ref.rollout.name=vllm
actor_rollout_ref.rollout.tensor_model_parallel_size=1
actor_rollout_ref.rollout.n=8
```

This means:

- use vLLM for generation
- keep rollout on one GPU
- generate 8 responses per prompt


In [ ]:
try:
    import vllm
    print("vLLM import check: OK")
    print("vLLM version:", getattr(vllm, "__version__", "unknown"))
except Exception as e:
    print("vLLM import failed:")
    print(e)

## Step 5: Prepare the GSM8K Dataset

We use GSM8K as a simple reasoning-style dataset.

VERL provides a preprocessing script that converts the dataset into parquet files for training and validation.


In [ ]:
%%bash
set -e

cd "${HOME}/verl-workshop/verl"

python3 examples/data_preprocess/gsm8k.py --local_dir ../data/gsm8k

echo "Generated files:"
ls -lh ../data/gsm8k

## Step 6: Define Model and Dataset Paths

We use Qwen3-4B as the base model.

During LoRA training, the base model remains frozen, while small adapter weights are updated.


In [ ]:
from pathlib import Path

MODEL_PATH = "Qwen/Qwen3-4B"
WORKSHOP_DIR = Path.home() / "verl-workshop"
VERL_DIR = WORKSHOP_DIR / "verl"
DATA_DIR = WORKSHOP_DIR / "data" / "gsm8k"

train_files = str(DATA_DIR / "train.parquet")
test_files = str(DATA_DIR / "test.parquet")

print("MODEL_PATH:", MODEL_PATH)
print("VERL_DIR:", VERL_DIR)
print("train_files:", train_files)
print("test_files:", test_files)
print("Train file exists:", Path(train_files).exists())
print("Test file exists:", Path(test_files).exists())


## Pre-download the Qwen3-4B Model

This step downloads the model weights into the Hugging Face cache.

You can skip this cell if the model is already cached on your machine.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

print("Downloading model config / weights...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="cpu",
)

print("Model download check: OK")
del model

## Step 7: Understand the Key Training Configuration

The most important parts of the training command are:

| Config | Meaning |
|---|---|
| `algorithm.adv_estimator=grpo` | Use GRPO as the RL algorithm |
| `actor_rollout_ref.model.lora_rank=32` | Enable LoRA adapter training |
| `actor_rollout_ref.model.lora_alpha=64` | Set LoRA scaling |
| `actor_rollout_ref.rollout.name=vllm` | Use vLLM as the rollout backend |
| `actor_rollout_ref.rollout.n=8` | Generate 8 responses per prompt |
| `actor_rollout_ref.rollout.tensor_model_parallel_size=1` | Single-GPU rollout setup |
| `actor_rollout_ref.rollout.gpu_memory_utilization=0.5` | Limit vLLM GPU memory usage |
| `actor_rollout_ref.actor.strategy=fsdp2` | Use FSDP2 for the actor side |
| `actor_rollout_ref.ref.strategy=fsdp2` | Use FSDP2 for the reference side |

The training loop is:

```text
Prompt batch
  ↓
vLLM generates 8 responses per prompt
  ↓
Reward function scores the responses
  ↓
GRPO computes relative advantages
  ↓
VERL updates the LoRA adapters
  ↓
Next iteration
```


## Step 8: Launch Qwen3-4B LoRA GRPO Training

This cell launches the training job.

For workshop time control, this configuration is intentionally small and simple.

You can reduce the runtime further by lowering:

- `trainer.total_epochs`
- `trainer.test_freq`
- `data.train_batch_size`
- `actor_rollout_ref.rollout.n`
- `data.max_response_length`

Important:

- The notebook runs on a single GPU.
- vLLM is used for rollout generation.
- LoRA updates only adapter weights.
- GRPO performs reward-based policy optimization.


In [ ]:
%%bash
set -e

cd "${HOME}/verl-workshop/verl"

MODEL_PATH="Qwen/Qwen3-4B"
train_files="${HOME}/verl-workshop/data/gsm8k/train.parquet"
test_files="${HOME}/verl-workshop/data/gsm8k/test.parquet"

GPU_MEMORY_UTILIZATION=0.5
max_token_len_per_gpu=24576

# Single-GPU workshop configuration
TP_VALUE=1
TRAIN_BATCH_SIZE=16
MINI_BATCH_SIZE=16

python3 -m verl.trainer.main_ppo \
    algorithm.adv_estimator=grpo \
    trainer.val_before_train=True \
    data.train_files="${train_files}" \
    data.val_files="${test_files}" \
    data.train_batch_size=${TRAIN_BATCH_SIZE} \
    data.max_prompt_length=1024 \
    data.max_response_length=1024 \
    data.filter_overlong_prompts=True \
    data.truncation='error' \
    data.shuffle=False \
    actor_rollout_ref.model.path="${MODEL_PATH}" \
    actor_rollout_ref.model.use_remove_padding=True \
    actor_rollout_ref.model.enable_gradient_checkpointing=True \
    actor_rollout_ref.model.lora_rank=32 \
    actor_rollout_ref.model.lora_alpha=64 \
    actor_rollout_ref.actor.optim.lr=1.0e-05 \
    actor_rollout_ref.actor.use_dynamic_bsz=True \
    actor_rollout_ref.actor.ppo_mini_batch_size=${MINI_BATCH_SIZE} \
    actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${max_token_len_per_gpu} \
    actor_rollout_ref.actor.use_kl_loss=True \
    actor_rollout_ref.actor.kl_loss_coef=0.001 \
    actor_rollout_ref.actor.kl_loss_type=low_var_kl \
    actor_rollout_ref.actor.entropy_coeff=0 \
    actor_rollout_ref.actor.strategy=fsdp2 \
    actor_rollout_ref.actor.fsdp_config.model_dtype=bf16 \
    actor_rollout_ref.actor.fsdp_config.param_offload=False \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=False \
    actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True \
    actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
    actor_rollout_ref.rollout.tensor_model_parallel_size=${TP_VALUE} \
    actor_rollout_ref.rollout.name=vllm \
    actor_rollout_ref.rollout.gpu_memory_utilization=${GPU_MEMORY_UTILIZATION} \
    actor_rollout_ref.rollout.n=8 \
    actor_rollout_ref.rollout.load_format=safetensors \
    actor_rollout_ref.rollout.layered_summon=True \
    actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True \
    actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
    actor_rollout_ref.ref.strategy=fsdp2 \
    actor_rollout_ref.ref.fsdp_config.model_dtype=bf16 \
    algorithm.use_kl_in_reward=False \
    trainer.use_legacy_worker_impl=disable \
    trainer.critic_warmup=0 \
    trainer.logger='["console","tensorboard"]' \
    trainer.project_name='grpo_qwen_llm' \
    trainer.experiment_name='qwen3_4b_grpo_lora_rocm_single_gpu' \
    trainer.n_gpus_per_node=1 \
    trainer.nnodes=1 \
    trainer.save_freq=-1 \
    trainer.test_freq=10 \
    trainer.total_epochs=1

## What This Training Job Launches

This launches a full RL pipeline:

- Ray worker initialization
- actor model setup
- reference model setup
- vLLM rollout workers
- GRPO training loop
- on-policy rollout → reward → train → update cycle
- TensorBoard logging

The most important runtime relationship is:

```text
VERL = training orchestration
vLLM = rollout generation
GRPO = reward-based policy optimization
LoRA = lightweight policy update
MI300 = single-GPU hardware foundation
```


## Step 9: Monitor Training Logs

The training command logs to the console and TensorBoard.

Typical signals to watch:

- rollout generation speed
- reward value
- response length
- KL value
- policy loss
- entropy
- validation score

If the workshop is time-limited, we only need to confirm that the pipeline runs and produces a few training iterations.


In [ ]:
%%bash
set -e

cd "${HOME}/verl-workshop/verl"

echo "Possible TensorBoard log locations:"
find . -maxdepth 4 -type d \( -name "tensorboard" -o -name "runs" -o -name "logs" \) 2>/dev/null || true

echo ""
echo "To start TensorBoard manually, you can run:"
echo "tensorboard --logdir . --host 0.0.0.0 --port 6006"